In [ ]:
# @title Imports and Notebook Utilities
# This block is mostly taken from the self-org textures notebook.
import os
import io
import PIL.Image, PIL.ImageDraw
import base64
import zipfile
import json
import requests
import numpy as np
import matplotlib.pylab as pl
import glob

from IPython.display import Image, HTML, Markdown, clear_output
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

os.environ['FFMPEG_BINARY'] = 'ffmpeg'
import moviepy.editor as mvp
from moviepy.video.io.ffmpeg_writer import FFMPEG_VideoWriter


def imread(url, max_size=None, mode=None):
    if isinstance(url, str) and url.startswith(('http:', 'https:')):
        # wikimedia requires a user agent
        headers = {
            "User-Agent": "Requests in Colab/0.0 (https://colab.research.google.com/; no-reply@google.com) requests/0.0"
        }
        r = requests.get(url, headers=headers)
        f = io.BytesIO(r.content)
    else:
        f = url
    img = PIL.Image.open(f)
    if max_size is not None:
        img.thumbnail((max_size, max_size), PIL.Image.LANCZOS)
    if mode is not None:
        img = img.convert(mode)
    img = np.float32(img) / 255.0
    return img


def np2pil(a):
    if a.dtype in [np.float32, np.float64]:
        a = np.uint8(np.clip(a, 0, 1) * 255)
    return PIL.Image.fromarray(a)


def imwrite(f, a, fmt=None):
    a = np.asarray(a)
    if isinstance(f, str):
        fmt = f.rsplit('.', 1)[-1].lower()
        if fmt == 'jpg':
            fmt = 'jpeg'
        f = open(f, 'wb')
    np2pil(a).save(f, fmt, quality=95)


def imencode(a, fmt='jpeg'):
    a = np.asarray(a)
    if len(a.shape) == 3 and a.shape[-1] == 4:
        fmt = 'png'
    f = io.BytesIO()
    imwrite(f, a, fmt)
    return f.getvalue()


def im2url(a, fmt='jpeg'):
    encoded = imencode(a, fmt)
    base64_byte_string = base64.b64encode(encoded).decode('ascii')
    return 'data:image/' + fmt.upper() + ';base64,' + base64_byte_string


def imshow(a, fmt='jpeg', id=None):
    return display(Image(data=imencode(a, fmt)), display_id=id)


def grab_plot(close=True):
    """Return the current Matplotlib figure as an image"""
    fig = pl.gcf()
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer._renderer)
    a = np.float32(img[..., 3:] / 255.0)
    img = np.uint8(255 * (1.0 - a) + img[..., :3] * a)  # alpha
    if close:
        pl.close()
    return img



def zoom(img, scale=4):
    img = np.repeat(img, scale, 0)
    img = np.repeat(img, scale, 1)
    return img


class VideoWriter:
    def __init__(self, filename='_autoplay.mp4', fps=30.0, **kw):
        self.writer = None
        self.params = dict(filename=filename, fps=fps, **kw)

    def add(self, img):
        img = np.asarray(img)
        if self.writer is None:
            h, w = img.shape[:2]
            self.writer = FFMPEG_VideoWriter(size=(w, h), **self.params)
        if img.dtype in [np.float32, np.float64]:
            img = np.uint8(img.clip(0, 1) * 255)
        if len(img.shape) == 2:
            img = np.repeat(img[..., None], 3, -1)
        self.writer.write_frame(img)

    def close(self):
        if self.writer:
            self.writer.close()

    def __enter__(self):
        return self

    def __exit__(self, *kw):
        self.close()
        if self.params['filename'] == '_autoplay.mp4':
            self.show()

    def show(self, **kw):
        self.close()
        fn = self.params['filename']
        display(mvp.ipython_display(fn, **kw))

!nvidia-smi -L

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
import torch
import torchvision.models as models

torch.set_default_tensor_type('torch.cuda.FloatTensor')

TypeError: type torch.cuda.FloatTensor not available. Torch not compiled with CUDA enabled.

In [ ]:
#@title Loss Function Selection and Definitions
import torch.nn.functional as F

# Choose which loss function to use
loss_type = "sliced_ot"  #@param ["sliced_ot", "relaxed_ot"]

print(f"Selected loss function: {loss_type}")

# ============================================================================
# Sliced OT Loss 
# ============================================================================
def calc_styles_vgg(imgs, vgg):
    style_layers = [1, 6, 11, 18, 25]
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    x = (imgs - mean) / std
    b, c, h, w = x.shape
    features = [x.reshape(b, c, h * w)]
    for i, layer in enumerate(vgg[:max(style_layers) + 1]):
        x = layer(x)
        if i in style_layers:
            b, c, h, w = x.shape
            features.append(x.reshape(b, c, h * w))
    return features

def project_sort(x, proj):
    return torch.einsum('bcn,cp->bpn', x, proj).sort()[0]

def ot_loss(source, target, proj_n=32):
    ch, n = source.shape[-2:]
    projs = F.normalize(torch.randn(ch, proj_n), dim=0)
    source_proj = project_sort(source, projs)
    target_proj = project_sort(target, projs)
    target_interp = F.interpolate(target_proj, n, mode='nearest')
    return (source_proj - target_interp).square().sum()

def create_sliced_ot_loss(vgg, target_styles):
    """Create Sliced OT loss with rotation invariance."""
    def loss_f(imgs):
        source_features = calc_styles_vgg(imgs, vgg)
        min_loss = None
        for target_features in target_styles:
            loss = sum(ot_loss(x, y) for x, y in zip(source_features, target_features))
            if min_loss is None:
                min_loss = loss
            else:
                min_loss = torch.minimum(min_loss, loss)
        return min_loss
    return loss_f

# ============================================================================
# Relaxed OT Loss
# ============================================================================
class RelaxedOTLoss(torch.nn.Module):
    """https://arxiv.org/abs/1904.12785"""
    def __init__(self, vgg, target_styles, n_samples=1024):
        super().__init__()
        self.n_samples = n_samples
        self.vgg = vgg
        self.target_styles = target_styles  # List of rotated target features

    def get_vgg_features(self, imgs):
        style_layers = [1, 6, 11, 18, 25]
        mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
        std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
        x = (imgs - mean) / std
        b, c, h, w = x.shape
        features = [x.reshape(b, c, h * w)]
        for i, layer in enumerate(self.vgg[:max(style_layers) + 1]):
            x = layer(x)
            if i in style_layers:
                b, c, h, w = x.shape
                features.append(x.reshape(b, c, h * w))
        return features

    @staticmethod
    def pairwise_distances_cos(x, y):
        x_norm = torch.norm(x, dim=2, keepdim=True)  # (b, n, 1)
        y_t = y.transpose(1, 2)  # (b, c, m)
        y_norm = torch.norm(y_t, dim=1, keepdim=True)  # (b, 1, m)
        dist = 1. - torch.matmul(x, y_t) / (x_norm * y_norm + 1e-10)  # (b, n, m)
        return dist

    @staticmethod
    def style_loss(x, y):
        pairwise_distance = RelaxedOTLoss.pairwise_distances_cos(x, y)
        m1, m1_inds = pairwise_distance.min(1)
        m2, m2_inds = pairwise_distance.min(2)
        remd = torch.max(m1.mean(dim=1), m2.mean(dim=1))
        return remd

    @staticmethod
    def moment_loss(x, y):
        mu_x, mu_y = torch.mean(x, 1, keepdim=True), torch.mean(y, 1, keepdim=True)
        mu_diff = torch.abs(mu_x - mu_y).mean(dim=(1, 2))

        x_c, y_c = x - mu_x, y - mu_y
        x_cov = torch.matmul(x_c.transpose(1, 2), x_c) / (x.shape[1] - 1)
        y_cov = torch.matmul(y_c.transpose(1, 2), y_c) / (y.shape[1] - 1)

        cov_diff = torch.abs(x_cov - y_cov).mean(dim=(1, 2))
        return mu_diff + cov_diff

    def forward(self, generated_image):
        generated_features = self.get_vgg_features(generated_image)
        
        # Compute loss for each rotation and take minimum
        min_loss = None
        for target_features in self.target_styles:
            loss = 0.0
            # Iterate over the VGG layers
            for x, y in zip(generated_features, target_features):
                # x: generated features [b_x, c, n_x]
                # y: target features [b_y, c, n_y]
                b_x, c_x, n_x = x.shape
                b_y, c_y, n_y = y.shape
                
                # Sample spatial positions (not channels)
                n_samples = min(n_x, n_y, self.n_samples)
                
                # Sample from generated features
                indices_x = torch.argsort(torch.rand(b_x, 1, n_x, device=x.device), dim=-1)[..., :n_samples]
                x_sampled = x.gather(-1, indices_x.expand(b_x, c_x, n_samples))
                
                # Sample from target features
                indices_y = torch.argsort(torch.rand(b_y, 1, n_y, device=y.device), dim=-1)[..., :n_samples]
                y_sampled = y.gather(-1, indices_y.expand(b_y, c_y, n_samples))
                
                # Transpose to (b, n_samples, c) for loss computation
                x_sampled = x_sampled.transpose(1, 2)
                y_sampled = y_sampled.transpose(1, 2)
                
                loss += self.style_loss(x_sampled, y_sampled) + self.moment_loss(x_sampled, y_sampled)
            
            if min_loss is None:
                min_loss = loss.mean()
            else:
                min_loss = torch.minimum(min_loss, loss.mean())
        
        return min_loss

print("Loss functions defined successfully!")

In [ ]:
#@title Load VGG and Target image {vertical-output: true}
from scipy import ndimage
import io  # Add this import for BytesIO

vgg = models.vgg16(weights='IMAGENET1K_V1').features

from google.colab import files

print("Please upload your style image:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f'Using uploaded file: "{filename}"')

# Create BytesIO object and ensure it's at the start
img_bytes = io.BytesIO(uploaded[filename])
img_bytes.seek(0)  # Reset to beginning of file

# Convert to RGB to ensure 3 channels (handles RGBA, grayscale, etc.)
style_img = imread(img_bytes, max_size=128, mode='RGB')
# Ensure exactly 3 channels
if style_img.shape[-1] != 3:
    print(f"Warning: Image has {style_img.shape[-1]} channels, converting to RGB")
    style_img = style_img[..., :3]
style_img_torch = torch.tensor(style_img).permute(2, 0, 1).unsqueeze(0)

# RD system is fully isotropic, so we use a variant of texture loss that
# tries to match input image with rotated versions of the target sample
print("Computing rotation-invariant loss targets (64 angles)...")
target_styles = []
for r in np.linspace(0.0, 360, 65)[:-1] + 0.12345:
    img_rotated = ndimage.rotate(style_img, r, reshape=False, mode='wrap')
    img_rotated_torch = torch.tensor(img_rotated).permute(2, 0, 1).unsqueeze(0)
    with torch.no_grad():
        style_features = calc_styles_vgg(img_rotated_torch, vgg)
        target_styles.append(style_features)

print(f"Precomputed {len(target_styles)} rotated style targets")

# Create loss function based on selected type
with torch.no_grad():
    if loss_type == "sliced_ot":
        print("Creating Sliced OT loss with rotation invariance...")
        loss_fn = create_sliced_ot_loss(vgg, target_styles)
        print("✓ Sliced OT loss initialized")
    elif loss_type == "relaxed_ot":
        print("Creating Relaxed OT loss with rotation invariance...")
        loss_fn = RelaxedOTLoss(vgg, target_styles, n_samples=1024)
        print("✓ Relaxed OT loss initialized")
    else:
        raise ValueError(f"Unknown loss type: {loss_type}")

imshow(style_img)

In [ ]:
#@title ReactionDiffusionCA Architecture
from scipy.ndimage import gaussian_filter

#@markdown ### Model Architecture
channel_n = 12  #@param {type: "integer"}

def laplacian(x):
    """Apply Laplacian filter to all channels independently."""
    b, ch, h, w = x.shape
    # normalized Laplacian (not applying normalization necessitates choosing smaller diffusion constants)
    lap = torch.tensor([[1.0, 2.0, 1.0],
                        [2.0, -12.0, 2.0],
                        [1.0, 2.0, 1.0]])/16. 

    # Depthwise convolution with circular padding
    y = x.reshape(b * ch, 1, h, w)
    y = torch.nn.functional.pad(y, [1, 1, 1, 1], "circular")
    y = torch.nn.functional.conv2d(y, lap[None, None])
    return y.reshape(b, ch, h, w)


class ReactionDiffusionCA(torch.nn.Module):
    def __init__(self, chn=12, hidden_n=128, noise_level=0.1):
        super().__init__()
        self.chn = chn
        self.register_buffer("noise_level", torch.tensor([noise_level]))

        # Reaction network (operates on state directly, not perception)
        # For RD: input is state (chn channels), output is update (chn channels)
        self.w1 = torch.nn.Conv2d(chn, hidden_n, 1, bias=True)
        self.w2 = torch.nn.Conv2d(hidden_n, chn, 1, bias=False)

        # Initialize weights 
        torch.nn.init.xavier_normal_(self.w1.weight, gain=0.1) 
        torch.nn.init.zeros_(self.w1.bias)
        torch.nn.init.zeros_(self.w2.weight)

        # Per-channel diffusion coefficients (multi-scale)
        n_groups = chn // 4
        diff_coef = torch.tensor([0.125, 0.25, 0.5, 1.]).repeat(n_groups) # diffusion constants 
        # Handle remainder channels
        if chn % 4 != 0:
            diff_coef = torch.cat([diff_coef, torch.ones(chn % 4) * 0.5])
        self.register_buffer("diff_coef", diff_coef)

    def forward(self, x, r=1.0, d=1.0, noise=None, dt=1.0, debug=False):
        """
        Args:
            x: State tensor [b, chn, h, w]
            r: Reaction rate scaling (default 1.0)
            d: Diffusion rate scaling (default 1.0)
            noise: Noise level to add (default None)
            dt: Time step for update (default 0.5)
            debug: Print diagnostic info (default False)
        """
        if debug:
            print(f"Input x: min={x.min().item():.4f}, max={x.max().item():.4f}, mean={x.mean().item():.4f}")

        # Add noise if specified
        if noise is not None and noise > 0:
            x = x + torch.randn_like(x) * noise
            if debug:
                print(f"After noise: min={x.min().item():.4f}, max={x.max().item():.4f}")

        # DIFFUSION TERM: Laplacian with per-channel coefficients
        diff = laplacian(x) * self.diff_coef[None, :, None, None]
        if debug:
            print(f"Diffusion: min={diff.min().item():.4f}, max={diff.max().item():.4f}")

        # REACTION TERM: Learned nonlinear dynamics with ReLU activation
        y = self.w1(x)
        if debug:
            print(f"After w1: min={y.min().item():.4f}, max={y.max().item():.4f}")
        y = torch.relu(y)  # ReLU activation (same as NoiseNCA)
        if debug:
            print(f"After relu: min={y.min().item():.4f}, max={y.max().item():.4f}")
        react = self.w2(y)
        if debug:
            print(f"Reaction: min={react.min().item():.4f}, max={react.max().item():.4f}")

        # Explicit reaction-diffusion update with time step
        x = x + dt * (diff * d + react * r)
        if debug:
            print(f"Output x: min={x.min().item():.4f}, max={x.max().item():.4f}")

        return x

    def seed(self, n, h=128, w=128, seed_type='uniform'):
        """Generate initial state.

        Args:
            n: Batch size
            h, w: Image dimensions
            seed_type: 'uniform' (default NCA-style) or 'gaussian_blobs' (RD-style)
        """
        if seed_type == 'gaussian_blobs':
            return self.seed_gaussian_blobs(n, h, w)
        else:
            # Default: uniform random noise
            return (torch.rand(n, self.chn, h, w) - 0.5) * self.noise_level

    def seed_gaussian_blobs(self, n, h=128, w=128, spot_prob=0.01, spread=3.0):
        """Create seed states with scattered gaussian blobs (from RD paper).

        This creates sparse random spots and blurs them with a Gaussian filter,
        creating smooth blob-like initial conditions. Only RGB channels are
        initialized; hidden channels start at 0.

        Args:
            n: Batch size
            h, w: Image dimensions
            spot_prob: Probability of a spot at each pixel (default 0.005)
            spread: Gaussian blur sigma (default 3.0)
        """
        # Create sparse random spots (only 0.5% of pixels are 1.0)
        x = np.floor(np.random.uniform(0, 1, (n, h, w, 1)) + spot_prob)

        # Blur with Gaussian filter (mode='wrap' for toroidal boundary)
        x = gaussian_filter(x, sigma=[0.0, spread, spread, 0.0], mode='wrap')

        # Scale by spread^2 to compensate for blur normalization
        x = x * spread ** 2

        # Replicate to RGB channels (3 channels)
        x = np.repeat(x, 3, axis=-1)

        # Pad with zeros for remaining hidden channels
        x = np.pad(x, [(0, 0), (0, 0), (0, 0), (0, self.chn - 3)])

        # Convert to PyTorch tensor and permute to [n, chn, h, w]
        return torch.tensor(x, dtype=torch.float32).permute(0, 3, 1, 2)


def to_rgb(s):
    """Extract RGB channels from state."""
    return s[..., :3, :, :] + 0.5


# Initialize model with selected channel count
model = ReactionDiffusionCA(chn=channel_n)
param_n = sum(p.numel() for p in model.parameters())
print(f'ReactionDiffusionCA with {channel_n} channels')
print(f'Parameter count: {param_n}')

# Visualize both seed types
print('\nUniform noise seeds (NCA-style):')
img = to_rgb(model.seed(4, 128, seed_type='uniform'))
imshow(np.hstack(img.permute(0, 2, 3, 1).cpu().numpy()))

print('\nGaussian blob seeds (RD-style):')
img = to_rgb(model.seed(4, 128, seed_type='gaussian_blobs'))
imshow(np.hstack(img.permute(0, 2, 3, 1).cpu().numpy()))

In [ ]:
#@title Debug Test: Check for NaN in Forward Pass
print("Testing model forward pass with debug output...")
# Use model from architecture cell (already initialized with channel_n)
test_state = model.seed(1, 128, seed_type='uniform')
print(f"\nInitial seed state: min={test_state.min().item():.4f}, max={test_state.max().item():.4f}")

print("\n--- Testing 5 forward steps with debug=True ---")
for step in range(5):
    print(f"\n=== Step {step} ===")
    test_state = model(test_state, r=1.0, d=1.0, noise=0.0, dt=1.0, debug=True)
    if torch.isnan(test_state).any():
        print(f"ERROR: NaN detected at step {step}!")
        break
    
print("\n--- Testing loss computation ---")
test_rgb = to_rgb(test_state)
print(f"RGB output: min={test_rgb.min().item():.4f}, max={test_rgb.max().item():.4f}")
print(f"Contains NaN: {torch.isnan(test_rgb).any().item()}")

# Only test loss if no NaN
if not torch.isnan(test_rgb).any():
    print("Computing loss...")
    with torch.no_grad():
        test_loss = loss_fn(test_rgb)
        print(f"Loss value: {test_loss.item():.4e}")
        print(f"Loss is NaN: {torch.isnan(test_loss).item()}")
else:
    print("Skipping loss computation due to NaN in RGB output")
    
print("\nDiagnostic test complete!")

In [ ]:
#@title Setup Training
import os
from google.colab import files

# Check for checkpoint files in Colab storage
checkpoint_files = glob.glob('*.pt')

if checkpoint_files:
    print(f"Found {len(checkpoint_files)} checkpoint file(s) in Colab storage:")
    for i, f in enumerate(checkpoint_files):
        file_size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  [{i}] {f} ({file_size_mb:.2f} MB)")

    choice = input("\nEnter number to load, 'u' to upload, or Enter for fresh start: ").strip()

    if choice == 'u':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        print(f'Loading checkpoint from uploaded file: "{filename}"')
        checkpoint = torch.load(filename)
    elif choice.isdigit() and 0 <= int(choice) < len(checkpoint_files):
        filename = checkpoint_files[int(choice)]
        print(f'Loading checkpoint from: "{filename}"')
        checkpoint = torch.load(filename)
    else:
        checkpoint = None
else:
    print("No checkpoint files found in Colab storage.")
    upload_choice = input("Upload a checkpoint file? (y/n, default=n): ").strip().lower()

    if upload_choice == 'y':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        print(f'Loading checkpoint from: "{filename}"')
        checkpoint = torch.load(filename)
    else:
        checkpoint = None

# Use model from architecture cell (already initialized with channel_n)
if checkpoint is not None:
    model.load_state_dict(checkpoint['model_state_dict'])
    start_iter = checkpoint.get('iteration', 0)
    print(f'Loaded checkpoint from iteration {start_iter}')
else:
    print(f'Using freshly initialized ReactionDiffusionCA model with {channel_n} channels')
    start_iter = 0

# Optimizer
opt = torch.optim.Adam(model.parameters(), 1e-3, capturable=True)

# Adaptive learning rate scheduler
lr_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt,
    mode='min',
    factor=0.2,
    patience=1000,
    threshold=0.01,
    threshold_mode='rel',
    min_lr=1e-6
)

# Load optimizer and scheduler state if resuming
if checkpoint is not None:
    if 'optimizer_state_dict' in checkpoint:
        opt.load_state_dict(checkpoint['optimizer_state_dict'])
    if 'scheduler_state_dict' in checkpoint:
        lr_sched.load_state_dict(checkpoint['scheduler_state_dict'])
    if 'loss_log' in checkpoint:
        loss_log = checkpoint['loss_log']
    else:
        loss_log = []
    if 'pool' in checkpoint:
        pool = checkpoint['pool']
    else:
        with torch.no_grad():
            pool = model.seed(256)
else:
    loss_log = []
    with torch.no_grad():
        pool = model.seed(256)

print(f'Starting from iteration {start_iter}')
print(f'Pool shape: {pool.shape}')

In [ ]:
# @title Training loop {vertical-output: true}

# Training parameters
num_iterations = 15000  #@param {type: "integer"}
reaction_rate = 1.0    #@param {type: "number"}
diffusion_rate = 1.0   #@param {type: "number"}
noise_level = 0.1     #@param {type: "number"}
dt = 0.5              #@param {type: "number"}
seed_type = "gaussian_blobs"  #@param ["uniform", "gaussian_blobs"]

# Reinitialize pool with selected seed type if starting fresh
if start_iter == 0:
    print(f"Initializing pool with seed_type='{seed_type}'")
    with torch.no_grad():
        pool = model.seed(256, seed_type=seed_type)

try:
    for i in range(start_iter, start_iter + num_iterations):
        with torch.no_grad():
            batch_idx = np.random.choice(len(pool), 4, replace=False)
            s = pool[batch_idx]
            if i % 32 == 0: # injecting seeds into the pool less often 
                s[:1] = model.seed(1, seed_type=seed_type)

        # Random number of steps
        step_n = np.random.randint(32, 96)
        for k in range(step_n):
            s = model(s, r=reaction_rate, d=diffusion_rate, noise=noise_level, dt=dt)

        # Compute loss
        overflow_loss = (s - s.clamp(-1.0, 1.0)).abs().sum()
        loss = loss_fn(to_rgb(s)) + overflow_loss

        # Backward pass and optimize (MUST be outside no_grad!)
        loss.backward()
        for p in model.parameters():
            p.grad /= (p.grad.norm() + 1e-8)  # Normalize gradients
        opt.step()
        opt.zero_grad()
        
        with torch.no_grad():
            lr_sched.step(loss)  # Adaptive scheduler
            pool[batch_idx] = s.detach()

            loss_log.append(loss.item())

            # Display progress
            if i % 5 == 0:
                current_lr = opt.param_groups[0]['lr']
                display(Markdown(f'''
            iteration: {i}
            loss: {loss.item():.2e}
            lr: {current_lr:.2e}'''), display_id='stats')

            # Plot and visualize
            if i % 24 == 0:
                pl.plot(loss_log, '.', alpha=0.1)
                pl.yscale('log')
                pl.ylim(np.min(loss_log), loss_log[0])
                pl.tight_layout()
                imshow(grab_plot(), id='log')
                imgs = to_rgb(s).permute([0, 2, 3, 1]).cpu()
                imshow(np.hstack(imgs), id='batch')

            # Save full checkpoint every 5000 iterations
            if i % 5000 == 0 and i > start_iter:
                checkpoint = {
                    'iteration': i,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': opt.state_dict(),
                    'scheduler_state_dict': lr_sched.state_dict(),
                    'loss_log': loss_log,
                    'pool': pool,
                }
                torch.save(checkpoint, f'rd_checkpoint_iter_{i}.pt')
                print(f'\nSaved full checkpoint at iteration {i}')

except KeyboardInterrupt:
    print('\n\nTraining interrupted by user!')
    print(f'Saving checkpoint at iteration {i}...')
    checkpoint = {
        'iteration': i,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'scheduler_state_dict': lr_sched.state_dict(),
        'loss_log': loss_log,
        'pool': pool,
    }
    torch.save(checkpoint, f'rd_checkpoint_interrupted_iter_{i}.pt')
    print(f'Checkpoint saved as rd_checkpoint_interrupted_iter_{i}.pt')
    print('You can now proceed to the next cell to download weights.')

print(f'\nTraining completed at iteration {i}')

In [ ]:
#@title Save Model Weights
import torch
from google.colab import files

# Save weights-only file (for demo) - this will be downloaded
# Include training hyperparameters so the demo can use the exact same values
weights_filename = 'rd_weights.pt'

# Get the model's state dict and add training hyperparameters
weights_dict = model.state_dict()

# Add training hyperparameters as special keys (prefixed with underscore)
weights_dict['_training_params'] = {
    'model_type': 'rd',
    'dt': dt,
    'noise_level': noise_level,
    'diff_coef': model.diff_coef.cpu().tolist(),
    'reaction_rate': reaction_rate,
    'diffusion_rate': diffusion_rate,
    'seed_type': seed_type,
    'channel_n': channel_n,
}

torch.save(weights_dict, weights_filename)
print(f"Model weights saved as '{weights_filename}'")
print(f"  Training hyperparameters included:")
print(f"    - model_type: rd")
print(f"    - channel_n: {channel_n}")
print(f"    - dt: {dt}")
print(f"    - noise_level: {noise_level}")
print(f"    - diff_coef: {model.diff_coef.cpu().tolist()}")
print(f"    - reaction_rate: {reaction_rate}")
print(f"    - diffusion_rate: {diffusion_rate}")
print(f"    - seed_type: {seed_type}")

# Save full checkpoint (for resuming training) - stays in Colab storage
checkpoint_filename = f'rd_checkpoint_final.pt'
checkpoint = {
    'iteration': i,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': opt.state_dict(),
    'scheduler_state_dict': lr_sched.state_dict(),
    'loss_log': loss_log,
    'pool': pool,
    'training_params': weights_dict['_training_params'],  # Also save in checkpoint
    'channel_n': channel_n,
}
torch.save(checkpoint, checkpoint_filename)
print(f"\nFull checkpoint saved as '{checkpoint_filename}' (in Colab storage)")

# Only download the weights file
print("\nDownloading weights file...")
files.download(weights_filename)